# Lab 0: Watch ACT control a robot

Run each cell from top to bottom. You will load a supplied ACT checkpoint, reset one LIBERO-Goal task, run a live rollout, and watch its video. This lab uses development start states only. A task name selects the matching simulator task and checkpoint. Standard ACT does not read the task name as a language command.

Start Jupyter from the repository root with this project's `.venv` kernel. If setup fails, run `robot-lab doctor` in a terminal. [Lab instructions](README.md) give the terminal command.

<a id="exercise"></a>

## Exercise: Test a second start state

1. Run the reference cells in `00-first-run.ipynb`.
2. Change `STATE_ID` from `0` to a different development state in `0..19`.
3. Run the notebook again. Compare the first image, the number of steps, and the `success` value.
4. Write down one visible action that differs between the two runs. Do not change the checkpoint or the final state list.

The [solution guide](../../solutions/00-first-run.md) gives one way to inspect the two run records. Complete your own comparison before you read it.


## 1. Select the task and start state

Choose `bowl_plate` or `bowl_stove`. The supplied manifest pairs each task with its ACT checkpoint. Change `STATE_ID` only within `0..19`. States `20..49` are held for the final evaluation.

In [ ]:
from pathlib import Path
import json
import tempfile
import yaml
from IPython.display import Video, display
from vla_lab.evaluation.demo import run_act_demo

def find_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


WORKSPACE = find_root()
TASK = "bowl_plate"  # Or "bowl_stove".
STATE_ID = 0  # Use a development state from 0 to 19.
assert 0 <= STATE_ID <= 19
references = yaml.safe_load((WORKSPACE / "manifests" / "act_references.yaml").read_text())
checkpoint = WORKSPACE / references["tasks"][TASK]["checkpoint_path"]
assert (checkpoint / "manifest.json").is_file(), "Download or restore the matching ACT checkpoint."
print(f"Task: {TASK} | state: {STATE_ID} | checkpoint: {checkpoint}")

## 2. Run the live policy

This cell resets the simulator to the selected start state. ACT reads camera images and robot state, predicts action chunks, and sends actions to the 20 Hz relative controller. It saves a video and a `run.json` record. The run can take several minutes.

In [ ]:
runs_root = WORKSPACE / ".local-data" / "runs"
runs_root.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="lab0-act-", dir=runs_root))
record = run_act_demo(
    task_alias=TASK,
    checkpoint=checkpoint,
    state_id=STATE_ID,
    configs_root=WORKSPACE / "configs",
    data_root=WORKSPACE / ".local-data",
    output_dir=run_dir,
)
print(json.dumps({key: record[key] for key in ("task_alias", "state_id", "success", "steps", "clipped_values", "elapsed_s")}, indent=2))
print(f"Run record: {run_dir / 'run.json'}")

## 3. Watch and inspect

The video shows the simulator view during the live run. `success` is the simulator's result for this one start state. One success or failure is not an estimate of overall performance.

In [ ]:
display(Video(record["video"], embed=True, html_attributes="controls"))
print(f"Checkpoint SHA-256: {record['checkpoint_model_sha256']}")
print(f"Video SHA-256: {record['video_sha256']}")

## 4. Reset and repeat

Run the same start state again. The second call opens a new simulator, resets it, and loads the same ACT checkpoint. It writes a separate run record. Compare the two results before you change the start state.

In [ ]:
repeat_dir = Path(tempfile.mkdtemp(prefix="lab0-act-reset-", dir=runs_root))
repeat = run_act_demo(
    task_alias=TASK,
    checkpoint=checkpoint,
    state_id=15,
    configs_root=WORKSPACE / "configs",
    data_root=WORKSPACE / ".local-data",
    output_dir=repeat_dir,
)
print(json.dumps({"first": {"success": record["success"], "steps": record["steps"]}, "reset": {"success": repeat["success"], "steps": repeat["steps"]}}, indent=2))
display(Video(repeat["video"], embed=True, html_attributes="controls"))

## Exercise

Follow [the exercise](#exercise): change only `STATE_ID` to another development state, run the notebook again, and compare the videos and run records. Keep each checkpoint paired with its task. Keep final states closed during model selection.